In [7]:
import pandas as pd
import numpy as np
import wandb
from IPython.display import display

In [8]:
ENTITY   = "bombaclat-mpi"
PROJECTS = {
    "persona":    "ma-steering-lora-nothink-v3",
    "nopersona":  "ma-steering-lora-nothink-v3-nopersona",
}
METRIC = "reward/raw_cos_sim_mean"

api = wandb.Api()

def fetch_runs(entity, project, label):
    rows = []
    for run in api.runs(f"{entity}/{project}"):
        hist = run.history(keys=[METRIC], x_axis="_step", pandas=False)
        vals = [(r["_step"], r[METRIC]) for r in hist if r.get(METRIC) is not None]
        if not vals:
            continue
        first_step, first_val = vals[0]
        last_step,  last_val  = vals[-1]
        rows.append({
            "condition":  label,
            "run_name":   run.name,
            "first_step": int(first_step),
            "first_val":  first_val,
            "last_step":  int(last_step),
            "last_val":   last_val,
            "delta":      last_val - first_val,
            "n_steps":    int(last_step),
        })
    return rows

raw = fetch_runs(ENTITY, PROJECTS["persona"],   "persona")
raw += fetch_runs(ENTITY, PROJECTS["nopersona"],"nopersona")

print(f"fetched {len(raw)} runs")

fetched 25 runs


In [9]:
# Parse trait + direction from run name
# e.g. ipdNV3_min_ethical  →  trait=ethical, dir=min
# e.g. ipdNV3np_max_adaptable_flexible  →  trait=adaptable flexible, dir=max

import re

def parse_name(run_name):
    m = re.match(r"ipdNV3(?:np)?_(max|min)_(.*)", run_name)
    if not m:
        return None, None
    direction = m.group(1)
    trait = m.group(2).replace("_", " ")
    return trait, direction

records = []
for r in raw:
    trait, direction = parse_name(r["run_name"])
    if trait is None:
        continue
    records.append({
        "trait":      trait,
        "dir":        direction,
        "condition":  r["condition"],
        "start":      r["first_val"],
        "end":        r["last_val"],
        "steps":      r["n_steps"],
        "delta":      r["delta"],
    })

flat = pd.DataFrame(records)

In [10]:
# Pivot: one row per (trait, dir), columns for persona and nopersona

p  = flat[flat.condition == "persona"].set_index(["trait", "dir"])[["start", "end", "steps", "delta"]]
np_ = flat[flat.condition == "nopersona"].set_index(["trait", "dir"])[["start", "end", "steps", "delta"]]

p.columns   = pd.MultiIndex.from_tuples([("Persona LoRA",    c) for c in p.columns])
np_.columns = pd.MultiIndex.from_tuples([("No-Persona LoRA", c) for c in np_.columns])

df = p.join(np_, how="outer")

# Sort by |persona delta| descending
df = df.sort_values(("Persona LoRA", "delta"), key=abs, ascending=False)

df.head

<bound method NDFrame.head of                        Persona LoRA                           No-Persona LoRA  \
                              start       end steps     delta           start   
trait              dir                                                          
angry              max    -0.146725 -0.064535    74  0.082190       -0.125121   
agreeableness      min    -0.010182  0.063591    75  0.073773       -0.020080   
adaptable flexible max    -0.181918 -0.133130   170  0.048788       -0.180232   
ethical            max    -0.077408 -0.046017   134  0.031391       -0.087691   
adaptable          min     0.133288  0.156290   152  0.023002        0.133484   
apathetic          max    -0.236814 -0.220010    28  0.016803             NaN   
agreeableness      max     0.025827  0.040714    84  0.014886        0.027454   
angry              min     0.145240  0.157367    73  0.012128             NaN   
adaptable          max    -0.128138 -0.135873   130 -0.007735       -0.125973  

In [11]:
def color_delta(val):
    if pd.isna(val):
        return "color: #aaa"
    if val > 0:
        intensity = min(abs(val) / 0.08, 1.0)
        alpha = 0.15 + 0.35 * intensity
        return "background-color: rgba(40,167,80,{:.2f}); color: #1a5c30; font-weight: 600".format(alpha)
    return "background-color: rgba(200,50,50,0.18); color: #7a1010; font-weight: 600"

delta_cols = [("Persona LoRA", "delta"), ("No-Persona LoRA", "delta")]
steps_cols = [("Persona LoRA", "steps"), ("No-Persona LoRA", "steps")]

fmt = {}
for cond in ["Persona LoRA", "No-Persona LoRA"]:
    fmt[(cond, "start")] = "{:+.4f}"
    fmt[(cond, "end")]   = "{:+.4f}"
    fmt[(cond, "steps")] = "{:.0f}"
    fmt[(cond, "delta")] = "{:+.4f}"

styled = (
    df.style
    .format(fmt, na_rep="—")
    .map(color_delta, subset=delta_cols)
    .set_table_styles([
        {"selector": "th",
         "props": "background: #f0f0f0; font-size: 12px; padding: 6px 10px; "
                  "border-bottom: 1px solid #ccc; text-align: center;"},
        {"selector": "th.col_heading.level0",
         "props": "font-size: 13px; font-weight: bold; border-bottom: 2px solid #888;"},
        {"selector": "th.row_heading",
         "props": "text-align: left; font-size: 12px;"},
        {"selector": "td",
         "props": "font-size: 12px; padding: 5px 10px; font-family: monospace; text-align: right;"},
        {"selector": "tr:hover td",
         "props": "background: #f0f4ff;"},
    ])
    .set_caption(
        "reward/raw_cos_sim_mean — first step → last step   |   sorted by |Persona Δ| desc"
    )
)

display(styled)

In [12]:
# Summary
p_delta_mean  = df[("Persona LoRA",    "delta")].mean()
np_delta_mean = df[("No-Persona LoRA", "delta")].mean()
n_persona  = df[("Persona LoRA",    "delta")].notna().sum()
n_nopersona = df[("No-Persona LoRA", "delta")].notna().sum()

paired = df.dropna(subset=[("Persona LoRA", "delta"), ("No-Persona LoRA", "delta")])
p_wins  = (paired[("Persona LoRA", "delta")] > paired[("No-Persona LoRA", "delta")]).sum()
np_wins = (paired[("No-Persona LoRA", "delta")] > paired[("Persona LoRA", "delta")]).sum()

print("Persona    mean delta : {:+.4f}  (n={})".format(p_delta_mean,  n_persona))
print("No-Persona mean delta : {:+.4f}  (n={})".format(np_delta_mean, n_nopersona))
print()
print("Persona    wins (paired) : {} / {}".format(p_wins,  len(paired)))
print("No-Persona wins (paired) : {} / {}".format(np_wins, len(paired)))

Persona    mean delta : +0.0220  (n=14)
No-Persona mean delta : +0.0171  (n=11)

Persona    wins (paired) : 7 / 11
No-Persona wins (paired) : 4 / 11
